In [1]:
import pprint
from typing import Any, Dict
import pandas as pd
from langchain_classic.output_parsers import PandasDataFrameOutputParser
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableLambda
from langchain_openai import ChatOpenAI
from langchain_teddynote import logging
from dotenv import load_dotenv

load_dotenv()
logging.langsmith("CH03-Pandas-DF-OutputParser")

LangSmith 추적을 시작합니다.
[프로젝트명]
CH03-Pandas-DF-OutputParser


In [2]:
model = ChatOpenAI(temperature=0, model="gpt-4o-mini")

In [3]:
def format_parser_output(parser_output: Dict[str, Any]) -> None:
    for key in parser_output.keys():
        parser_output[key] = parser_output[key].to_dict()
    return pprint.PrettyPrinter(width=4, compact=True).pprint(parser_output)

In [4]:
df = pd.read_csv("./data/titanic.csv")
df.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


In [5]:
parser = PandasDataFrameOutputParser(dataframe=df)
print(parser.get_format_instructions())


The output should be formatted as a string as the operation, followed by a colon, followed by the column or row to be queried on, followed by optional array parameters.
1. The column names are limited to the possible columns below.
2. Arrays must either be a comma-separated list of numbers formatted as [1,3,5], or it must be in range of numbers formatted as [0..4].
3. Remember that arrays are optional and not necessarily required.
4. If the column is not in the possible columns or the operation is not a valid Pandas DataFrame operation, return why it is invalid as a sentence starting with either "Invalid column" or "Invalid operation".

As an example, for the formats:
1. String "column:num_legs" is a well-formatted instance which gets the column num_legs, where num_legs is a possible column.
2. String "row:1" is a well-formatted instance which gets row 1.
3. String "column:num_legs[1,2]" is a well-formatted instance which gets the column num_legs for rows 1 and 2, where num_legs is a p

**gpt-4o-mini**의 경우 출력 값에 "" 이붙어서 출력이 안맞아 짐. 

In [6]:
df_query = "Age Column을 조회해 주세요."

prompt = PromptTemplate(
    template="Answer the user query.\n{format_instructions}\n{question}\n",
    input_variables=["question"],
    partial_variables={
        "format_instructions": parser.get_format_instructions()
    },
)


def clean_output(text: str) -> str:
    # gpt-4o-mini가 가끔 "column:Age" 처럼 앞뒤에 따옴표를 붙여서
    # PandasDataFrameOutputParser가 "column을 요청 타입으로 인식하지 못하는 문제 방지
    return text.strip().strip('"').strip("'")

chain = prompt | model | StrOutputParser() | RunnableLambda(clean_output) | parser
chain_2_comp = prompt | model | StrOutputParser() | RunnableLambda(clean_output)
chain_2_comp_2 = prompt | model | StrOutputParser() 
print("-"*20)
print(chain_2_comp.invoke({"question": df_query}))
print("-"*20)
print(chain_2_comp_2.invoke({"question": df_query}))

# gpt-4o-mini에서는 Error 발생
# chain = prompt | model | parser 
parser_output = chain.invoke({"question": df_query})

# format_parser_output(parser_output)


--------------------


column:Age
--------------------


"column:Age"


In [7]:
format_parser_output(parser_output)

{'Age': {0: 22.0,
         1: 38.0,
         2: 26.0,
         3: 35.0,
         4: 35.0,
         5: nan,
         6: 54.0,
         7: 2.0,
         8: 27.0,
         9: 14.0,
         10: 4.0,
         11: 58.0,
         12: 20.0,
         13: 39.0,
         14: 14.0,
         15: 55.0,
         16: 2.0,
         17: nan,
         18: 31.0,
         19: nan}}


In [8]:
df_query = "Retrieve the first row." #첫번째 행을 조회
parser_output = chain.invoke({"question":df_query})
format_parser_output(parser_output)

{'0': {'Age': 22.0,
       'Cabin': nan,
       'Embarked': 'S',
       'Fare': 7.25,
       'Name': 'Braund, '
               'Mr. '
               'Owen '
               'Harris',
       'Parch': 0,
       'PassengerId': 1,
       'Pclass': 3,
       'Sex': 'male',
       'SibSp': 1,
       'Survived': 0,
       'Ticket': 'A/5 '
                 '21171'}}


In [9]:
df["Age"].head().mean()

np.float64(31.2)

In [10]:
df_query = "Rerieve the average of the Ages from row 0 to 4."
parser_output = chain.invoke({"question": df_query})
print(parser_output)

{'mean': np.float64(31.2)}


In [11]:
df_query = "Calculate average 'Fare' rate." #통행 요금 전체 평균 
parser_output = chain.invoke({"question": df_query})
print(parser_output)

{'mean': np.float64(22.19937)}


In [12]:
df["Fare"].mean()

np.float64(22.19937)

## with_structured_output()으로 gpt-4o-mini 따옴표 문제 해결되는지 확인

`clean_output`으로 앞뒤 따옴표(`"column:Age"`)를 벗겨내는 방식 대신, `with_structured_output()`을 사용해 모델이 애초에 JSON(Pydantic) 스키마로만 답하도록 강제하면 따옴표 문제가 근본적으로 해결되는지 테스트.

In [13]:
from pydantic import BaseModel, Field


class PandasQuery(BaseModel):
    """PandasDataFrameOutputParser가 요구하는 형식의 쿼리 문자열"""

    query: str = Field(
        description=(
            "PandasDataFrameOutputParser 형식의 쿼리 문자열. "
            "예: 'column:Age', 'row:1', 'mean:Fare[0..4]' 처럼 "
            "따옴표 없이 순수한 쿼리 문자열만 담는다."
        )
    )


structured_model = model.with_structured_output(PandasQuery)

# StrOutputParser/clean_output 없이 바로 parser로 연결
chain_structured = prompt | structured_model | RunnableLambda(lambda x: x.query) | parser

df_query = "Age Column을 조회해 주세요."
raw_structured = (prompt | structured_model).invoke({"question": df_query})
print("구조화 출력 원본:", repr(raw_structured))

parser_output = chain_structured.invoke({"question": df_query})
format_parser_output(parser_output)

구조화 출력 원본: PandasQuery(query='column:Age')


{'Age': {0: 22.0,
         1: 38.0,
         2: 26.0,
         3: 35.0,
         4: 35.0,
         5: nan,
         6: 54.0,
         7: 2.0,
         8: 27.0,
         9: 14.0,
         10: 4.0,
         11: 58.0,
         12: 20.0,
         13: 39.0,
         14: 14.0,
         15: 55.0,
         16: 2.0,
         17: nan,
         18: 31.0,
         19: nan}}


In [14]:
# 나머지 케이스도 clean_output 없이 with_structured_output만으로 잘 동작하는지 확인
df_query = "Retrieve the first row."
parser_output = chain_structured.invoke({"question": df_query})
format_parser_output(parser_output)

df_query = "Rerieve the average of the Ages from row 0 to 4."
print(chain_structured.invoke({"question": df_query}))

df_query = "Calculate average 'Fare' rate."
print(chain_structured.invoke({"question": df_query}))

{'1': {'Age': 38.0,
       'Cabin': 'C85',
       'Embarked': 'C',
       'Fare': 71.2833,
       'Name': 'Cumings, '
               'Mrs. '
               'John '
               'Bradley '
               '(Florence '
               'Briggs '
               'Thayer)',
       'Parch': 0,
       'PassengerId': 2,
       'Pclass': 1,
       'Sex': 'female',
       'SibSp': 1,
       'Survived': 1,
       'Ticket': 'PC '
                 '17599'}}


{'mean': np.float64(31.2)}


{'mean': np.float64(22.19937)}


## PandasDataFrameOutputParser 없이, 현재 권장되는 방식으로 구현

`PandasDataFrameOutputParser`는 `langchain_classic`(구 langchain 커뮤니티/레거시 영역)에 있는 "문자열 미니 언어(`column:Age`, `mean:Fare[0..4]`...)를 만들고 그걸 다시 파싱"하는 구식 패턴이다.

현재 권장되는 방식은 이 문자열 미니 언어 자체를 없애고, `with_structured_output()`으로 **모델이 수행할 pandas 연산의 의도(operation/column/rows)를 곧바로 구조화된 필드로 뽑아내고**, 실제 실행은 순수 pandas 코드로 처리하는 것이다. 즉:

- (기존) LLM → 문자열("column:Age") → `PandasDataFrameOutputParser`가 문자열을 파싱해서 pandas 연산 수행
- (권장) LLM → 구조화된 객체(`operation="column", column="Age"`) → 우리 코드가 곧바로 pandas 연산 수행 (파싱 단계 자체가 불필요)

In [15]:
from typing import List, Literal, Optional

from pydantic import BaseModel, Field


class DataFrameQuery(BaseModel):
    """사용자 질문을 pandas 연산 의도로 구조화한 결과"""

    operation: Literal["column", "row", "mean", "sum", "min", "max", "std", "count"] = Field(
        description="수행할 pandas 연산. 'column'/'row'는 단순 조회, 나머지는 집계 연산."
    )
    column: Optional[str] = Field(
        default=None,
        description="연산 대상 컬럼명. row 전체 조회처럼 컬럼이 필요 없는 경우 생략.",
    )
    rows: Optional[List[int]] = Field(
        default=None,
        description="연산 대상 행 인덱스 목록. 생략하면 전체 행을 대상으로 함.",
    )


def execute_df_query(query: DataFrameQuery, dataframe: pd.DataFrame = df):
    """구조화된 쿼리를 실제 pandas 연산으로 실행 (문자열 파싱 단계 없음)"""
    target = dataframe if query.rows is None else dataframe.iloc[query.rows]

    if query.operation == "row":
        return target

    if query.column is None:
        raise ValueError(f"'{query.operation}' 연산에는 column이 필요합니다.")

    series = target[query.column]
    if query.operation == "column":
        return series
    return getattr(series, query.operation)()


query_prompt = PromptTemplate(
    template=(
        "다음은 분석 대상 데이터프레임의 컬럼 목록입니다: {columns}\n"
        "사용자 질문에 맞는 pandas 연산 정보를 구조화된 형태로 추출하세요.\n"
        "질문: {question}\n"
    ),
    input_variables=["question"],
    partial_variables={"columns": ", ".join(df.columns)},
)

structured_model_2 = ChatOpenAI(temperature=0, model="gpt-4o-mini").with_structured_output(
    DataFrameQuery
)

chain_no_parser = query_prompt | structured_model_2 | RunnableLambda(execute_df_query)

In [16]:
# 기존과 동일한 질의들을 파서 없이 그대로 테스트
for df_query in [
    "Age Column을 조회해 주세요.",
    "Retrieve the first row.",
    "Rerieve the average of the Ages from row 0 to 4.",
    "Calculate average 'Fare' rate.",
]:
    print("Q:", df_query)
    result = chain_no_parser.invoke({"question": df_query})
    print(result)
    print("-" * 40)

Q: Age Column을 조회해 주세요.


0     22.0
1     38.0
2     26.0
3     35.0
4     35.0
5      NaN
6     54.0
7      2.0
8     27.0
9     14.0
10     4.0
11    58.0
12    20.0
13    39.0
14    14.0
15    55.0
16     2.0
17     NaN
18    31.0
19     NaN
Name: Age, dtype: float64
----------------------------------------
Q: Retrieve the first row.


   PassengerId  Survived  Pclass                     Name   Sex   Age  SibSp  \
0            1         0       3  Braund, Mr. Owen Harris  male  22.0      1   

   Parch     Ticket  Fare Cabin Embarked  
0      0  A/5 21171  7.25   NaN        S  
----------------------------------------
Q: Rerieve the average of the Ages from row 0 to 4.


31.2
----------------------------------------
Q: Calculate average 'Fare' rate.


22.19937
----------------------------------------
